In [ ]:
# CashPred_project/
│
├── data/
│   ├── raw/
│   │   └── landsat_data.csv               # Raw Landsat 5, 7 & 8 data
│   ├── processed/
│   │   └── scaled_data.csv                # Preprocessed and scaled data
│   └── shap_values/                       # Optional: saved SHAP results
│
├── models/
│   ├── cnn_module.py                      # CNN block for feature extraction
│   ├── graphsage_dgnn.py                  # GraphSAGE + DGNN + fusion model
│   └── __init__.py
│
├── notebooks/
│   └── analysis_experiments.ipynb         # For experimental analysis and visualization
│
├── plots/
│   ├── heatmaps/
│   ├── bar_charts/
│   ├── shap/
│   └── performance/
│
├── utils/
│   ├── preprocessing.py                   # Imputation, scaling, feature selection
│   ├── evaluation.py                      # Metric functions (MSE, RMSE, etc.)
│   ├── visualization.py                   # Plotting heatmaps, bar charts, SHAP
│   └── graph_utils.py                     # For graph construction and plotting
│
├── main.py                                # Main script to run training & evaluation
├── requirements.txt                       # All pip packages
└── README.md                              # Project overview and usage

In [ ]:
pip install scikit-learn
pip install torch
pip install torch_geometric
pip install networkx

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import networkx as nx

# READ DATA
df = pd.read_csv("your_data.csv")  # Replace with your actual file path

# SCALING
scaler = MinMaxScaler()
features = ['ndvi', 'ndwi', 'lst', 'lai']
df[features] = scaler.fit_transform(df[features])

# CNN MODULE
class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(CNNBlock, self).__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.fc = nn.Linear(out_channels * (len(features) // 2), 50)

    def forward(self, x):
        x = x.unsqueeze(1)  # Add channel dimension: (B, 1, F)
        x = F.relu(self.conv(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc(x))
        return x

# GRAPHSAGE + DGNN MODULE
class GraphSAGE_DGNN(nn.Module):
    def __init__(self, in_feats, h_feats):
        super(GraphSAGE_DGNN, self).__init__()
        self.sage1 = SAGEConv(in_feats, h_feats, aggr="mean")
        self.sage2 = SAGEConv(h_feats, h_feats, aggr="mean")
        self.dgnn_layers = nn.ModuleList([nn.Linear(h_feats, h_feats) for _ in range(2)])
        self.fc = nn.Linear(h_feats, 1)

    def forward(self, x, edge_index):
        x = F.relu(self.sage1(x, edge_index))
        x = F.relu(self.sage2(x, edge_index))
        for layer in self.dgnn_layers:
            x = F.relu(layer(x))
        return self.fc(x).squeeze()


# CROSS-VALIDATION AND TRAINING
kf = KFold(n_splits=5)
results = []

X = torch.tensor(df[features].values, dtype=torch.float32)
y = torch.tensor(df['yield'].values, dtype=torch.float32)

# Dummy edge_index assuming full graph
edge_index = torch.combinations(torch.arange(X.size(0)), r=2).T

for train_idx, test_idx in kf.split(X):
    model = GraphSAGE_DGNN(in_feats=4, h_feats=16)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    for epoch in range(20):
        model.train()
        optimizer.zero_grad()
        out = model(X[train_idx], edge_index)
        loss = criterion(out, y[train_idx])
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = model(X[test_idx], edge_index)
        mse = mean_squared_error(y[test_idx], preds)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y[test_idx], preds)
        r2 = r2_score(y[test_idx], preds)
        results.append((rmse, r2, mse, mae))


# RELATIVE GAIN
avg = np.mean(results, axis=0)
print("Average Metrics: RMSE={:.3f}, R2={:.3f}, MSE={:.3f}, MAE={:.3f}".format(*avg))


# HEATMAP OF FEATURES
sns.heatmap(df[features + ['yield']].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()


# BAR GRAPH FOR VEGETATION INDICES
df[features].mean().plot(kind='bar', color='green')
plt.title("Average Vegetation Indices")
plt.ylabel("Scaled Value")
plt.show()

# MUNICIPAL PERFORMANCE PLOT
municipals = ['Jaman North', 'Jaman South', 'Wenchi']
municipal_metrics = {
    'RMSE': [],
    'R2':   [],
    'MSE':  [],
    'MAE':  [],
}

x = np.arange(len(municipals))
width = 0.2
fig, ax = plt.subplots(figsize=(10, 6))
for i, (metric, values) in enumerate(municipal_metrics.items()):
    ax.bar(x + i*width, values, width=width, label=metric)

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(municipals)
ax.set_title("Municipal Performance")
ax.legend()
plt.tight_layout()
plt.show()


# GRAPH VISUALIZATION (Simplified)
G = nx.Graph()
G.add_nodes_from(municipals)
G.add_edges_from([('Jaman North', 'Jaman South'), ('Jaman South', 'Wenchi')])
nx.draw(G, with_labels=True, node_color='lightblue', edge_color='gray')
plt.title("Graph Structure of Municipal Influence")
plt.show()

# SHAP PLOT (EXPLANATIONS)
model.eval()
explainer = shap.Explainer(model, X)
shap_values = explainer(X)
shap.summary_plot(shap_values, features=features, feature_names=features)
